## Exploración de estructura entre eras

Objetivo: Confirmar qué columnas cambian entre temporadas viejas y nuevas antes de escribir cualquier lógica de limpieza.

In [1]:
import pandas as pd

season_9394 = pd.read_csv('../data/raw/results/season-9394.csv')
season_0405 = pd.read_csv('../data/raw/results/season-0405.csv')
season_0506 = pd.read_csv('../data/raw/results/season-0506.csv')

for nombre, df in [('93-94', season_9394), ('04-05', season_0405), ('05-06', season_0506)]:
    print(nombre, df.shape)
    print(df.columns.tolist())
    print()

93-94 (380, 22)
['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR']

04-05 (380, 22)
['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR']

05-06 (380, 22)
['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR']



In [2]:
print('93-94 nulos en HS:', season_9394['HS'].isna().sum(), 'de', len(season_9394))
print('04-05 nulos en HS:', season_0405['HS'].isna().sum(), 'de', len(season_0405))
print('05-06 nulos en HS:', season_0506['HS'].isna().sum(), 'de', len(season_0506))

93-94 nulos en HS: 380 de 380
04-05 nulos en HS: 380 de 380
05-06 nulos en HS: 0 de 380


### Hallazgo: disponibilidad de estadísticas de juego por era

Las estadísticas de juego (tiros, tiros a puerta, córners, tarjetas, faltas: columnas `HS`, `AS`, `HST`, `AST`, `HF`, `AF`, `HC`, `AC`, `HY`, `AY`, `HR`, `AR`) solo están disponibles desde la temporada **2005/06** en adelante. En las temporadas 1993/94 a 2004/05, esas columnas existen en el CSV pero están 100% vacías.

**Implicación para el análisis:**
- Cualquier métrica de rendimiento basada en goles, resultado o puntos (`FTHG`, `FTAG`, `FTR`) cubre el rango completo, 1993/94 en adelante.
- Cualquier análisis que dependa de estadísticas de juego (posesión indirecta vía tiros, disciplina vía tarjetas, etc.) queda limitado a partir de 2005/06.

Esto no afecta la métrica principal de "temporada buena vs mala" definida más adelante (puntos, posición, diferencia de gol), que solo depende de goles y resultado.

In [3]:
equipo = 'Barcelona'

barca_9394 = season_9394[(season_9394['HomeTeam'] == equipo) | (season_9394['AwayTeam'] == equipo)]
barca_0405 = season_0405[(season_0405['HomeTeam'] == equipo) | (season_0405['AwayTeam'] == equipo)]
barca_0506 = season_0506[(season_0506['HomeTeam'] == equipo) | (season_0506['AwayTeam'] == equipo)]

for nombre, df in [('93-94', barca_9394), ('04-05', barca_0405), ('05-06', barca_0506)]:
    print(nombre, df.shape)

93-94 (38, 22)
04-05 (38, 22)
05-06 (38, 22)


### Reconstrucción de tabla de posiciones por temporada

Los CSVs de football-data.co.uk traen resultados partido a partido de toda la liga, no la tabla de posiciones ya calculada. Para saber en qué posición terminó el Barça cada temporada, hay que reconstruir la tabla completa de los 20 equipos (agregando puntos, victorias, empates, derrotas, goles a favor/contra y diferencia de gol) a partir de esos resultados.

**Decisión de diseño:** los puntos se calculan con la regla vigente en cada temporada (2 puntos por victoria antes de 1995/96, 3 puntos después), no normalizados. Esto respeta la clasificación real de cada año. Cualquier comparación de puntos entre una temporada de la era de 2 y una de la era de 3 se normaliza puntualmente en el momento de esa comparación específica, no en el dato base.

**Criterio de desempate:** diferencia de gol como único criterio secundario (los criterios oficiales de La Liga incluyen más factores como resultados entre los equipos empatados). Esto solo se valida manualmente contra la clasificación real cuando el desempate afecta una posición relevante (título, zona de Champions/Europa, descenso).

In [4]:
def construir_tabla_liga(df, temporada):
    puntos_victoria = 2 if temporada in ['9394', '9495'] else 3

    local = df[['HomeTeam', 'FTHG', 'FTAG']].copy()
    local.columns = ['equipo', 'goles_favor', 'goles_contra']

    visitante = df[['AwayTeam', 'FTAG', 'FTHG']].copy()
    visitante.columns = ['equipo', 'goles_favor', 'goles_contra']

    partidos = pd.concat([local, visitante], ignore_index=True)

    partidos['resultado'] = 'empate'
    partidos.loc[partidos['goles_favor'] > partidos['goles_contra'], 'resultado'] = 'victoria'
    partidos.loc[partidos['goles_favor'] < partidos['goles_contra'], 'resultado'] = 'derrota'

    partidos['puntos'] = partidos['resultado'].map({
        'victoria': puntos_victoria,
        'empate': 1,
        'derrota': 0
    })

    tabla = partidos.groupby('equipo').agg(
        partidos_jugados=('resultado', 'count'),
        victorias=('resultado', lambda x: (x == 'victoria').sum()),
        empates=('resultado', lambda x: (x == 'empate').sum()),
        derrotas=('resultado', lambda x: (x == 'derrota').sum()),
        goles_favor=('goles_favor', 'sum'),
        goles_contra=('goles_contra', 'sum'),
        puntos=('puntos', 'sum')
    ).reset_index()

    tabla['diferencia_gol'] = tabla['goles_favor'] - tabla['goles_contra']
    tabla = tabla.sort_values(['puntos', 'diferencia_gol'], ascending=False).reset_index(drop=True)
    tabla['posicion'] = tabla.index + 1

    return tabla

In [5]:
tabla_9394 = construir_tabla_liga(season_9394, '9394')
tabla_0405 = construir_tabla_liga(season_0405, '0405')
tabla_0506 = construir_tabla_liga(season_0506, '0506')

for nombre, tabla in [('93-94', tabla_9394), ('04-05', tabla_0405), ('05-06', tabla_0506)]:
    fila_barca = tabla[tabla['equipo'] == 'Barcelona']
    print(nombre, '- Posición:', fila_barca['posicion'].values[0], '- Puntos:', fila_barca['puntos'].values[0])

93-94 - Posición: 1 - Puntos: 56
04-05 - Posición: 1 - Puntos: 84
05-06 - Posición: 1 - Puntos: 82
